In [1]:
!git clone https://github.com/nttng207/TwinLiteNetPlus.git --branch develop/tantran

Cloning into 'TwinLiteNetPlus'...
remote: Enumerating objects: 180, done.
remote: Counting objects: 100% (87/87), done.
remote: Compressing objects: 100% (41/41), done.
remote: Total 180 (delta 65), reused 59 (delta 46), pack-reused 93 (from 1)
Receiving objects: 100% (180/180), 11.72 MiB | 25.54 MiB/s, done.
Resolving deltas: 100% (80/80), done.


In [2]:
!pip install -q gdown
!mkdir -p /kaggle/working/TwinLiteNetPlus/pretrained

!gdown --folder "https://drive.google.com/drive/folders/1EqBzUw0b17aEumZmWYrGZmbx_XJqU-vz" \
  -O /kaggle/working/TwinLiteNetPlus/pretrained


Retrieving folder contents
Processing file 1H8P-GrOUBOaVs5LEqXBfz0dC9gguUUio large.pth
Processing file 121z9XUh7_lgze8i6nS6Ne2HQ7ZG5_uG9 medium.pth
Processing file 1SvD03WZOq8eN4X0bMFNZyzj4upZM1QVl nano.pth
Processing file 1N7jNsa8P1dM-UevqPRsx6heUpN4L00FJ small.pth
Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1H8P-GrOUBOaVs5LEqXBfz0dC9gguUUio
To: /kaggle/working/TwinLiteNetPlus/pretrained/large.pth
100%|███████████████████████████████████████| 7.96M/7.96M [00:00<00:00, 139MB/s]
Downloading...
From: https://drive.google.com/uc?id=121z9XUh7_lgze8i6nS6Ne2HQ7ZG5_uG9
To: /kaggle/working/TwinLiteNetPlus/pretrained/medium.pth
100%|███████████████████████████████████████| 2.05M/2.05M [00:00<00:00, 216MB/s]
Downloading...
From: https://drive.google.com/uc?id=1SvD03WZOq8eN4X0bMFNZyzj4upZM1QVl
To: /kaggle/working/TwinLiteNetPlus/pretrained/nano.pth
100%|████████████████████████████████

In [3]:
!mkdir -p /kaggle/working/TwinLiteNetPlus/finetune

!gdown --folder "https://drive.google.com/drive/folders/11j_X43yxjpqsGh3rvodrHU5fb-EAX3eI" \
  -O /kaggle/working/TwinLiteNetPlus/finetune

Retrieving folder contents
Processing file 1CJ_cKUPQkB1EoTqMJLGXh6OUNZa0BVoi large.pth.tar
Processing file 17iedVkzbMR46wAgOAOC3nfyU1yPWYtcy medium.pth.tar
Processing file 1snCBNn_zrfc6V8tbVtcn4G3VNL7EyS8x nano.pth.tar
Processing file 11XehyfvpmXsNGK7ZQKXkvfVdFTbmbE7Z small.pth.tar
Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1CJ_cKUPQkB1EoTqMJLGXh6OUNZa0BVoi
To: /kaggle/working/TwinLiteNetPlus/finetune/large.pth.tar
100%|██████████████████████████████████████| 31.7M/31.7M [00:00<00:00, 58.6MB/s]
Downloading...
From: https://drive.google.com/uc?id=17iedVkzbMR46wAgOAOC3nfyU1yPWYtcy
To: /kaggle/working/TwinLiteNetPlus/finetune/medium.pth.tar
100%|██████████████████████████████████████| 8.14M/8.14M [00:00<00:00, 30.7MB/s]
Downloading...
From: https://drive.google.com/uc?id=1snCBNn_zrfc6V8tbVtcn4G3VNL7EyS8x
To: /kaggle/working/TwinLiteNetPlus/finetune/nano.pth.tar
100%|██████████

In [4]:
import torch
import os

base_dir = "/kaggle/working/TwinLiteNetPlus/finetune"

models = ["nano", "small", "medium", "large"]

for m in models:
    ckpt_path = os.path.join(base_dir, f"{m}.pth.tar")
    save_path = os.path.join(base_dir, f"{m}_weights.pth")

    ckpt = torch.load(ckpt_path, map_location="cpu")

    if "state_dict" in ckpt:
        state_dict = ckpt["state_dict"]
    elif "model" in ckpt:
        state_dict = ckpt["model"]
    else:
        state_dict = ckpt

    torch.save(state_dict, save_path)
    print(f"Saved {save_path}")

Saved /kaggle/working/TwinLiteNetPlus/finetune/nano_weights.pth
Saved /kaggle/working/TwinLiteNetPlus/finetune/small_weights.pth
Saved /kaggle/working/TwinLiteNetPlus/finetune/medium_weights.pth
Saved /kaggle/working/TwinLiteNetPlus/finetune/large_weights.pth


In [5]:
%cd /kaggle/working/TwinLiteNetPlus

/kaggle/working/TwinLiteNetPlus


# Model Version

## Nano

### Validate with Base

In [6]:
!python val_MAPILLARY.py --config 'nano' --weight '/kaggle/working/TwinLiteNetPlus/pretrained/nano.pth' \
                --data_root "/kaggle/input/datasets/kaggleprollc/mapillary-vistas-image-data-collection/Mapillary Vistas" \
                --num_workers 4 \
                --batch_size 16


Total network parameters: 33379
Driving Area Segment: mIOU(0.697)
Lane Line Segment: Acc(0.529) IOU(0.049)


### Finetuning

In [7]:
# !python train_MAPILLARY.py \
#     --data_root "/kaggle/input/datasets/kaggleprollc/mapillary-vistas-image-data-collection/Mapillary Vistas" \
#     --hyp "./hyperparameters/twinlitev2_hyper.yaml" \
#     --savedir "./finetune_mapillary/nano" \
#     --resume "/kaggle/working/TwinLiteNetPlus/pretrained/nano.pth" \
#     --config "nano" \
#     --max_epochs 40 \
#     --batch_size 32 \
#     --num_workers 4 \
#     --ema \
#     --verbose

### Validate with Finetuned

In [8]:
!python val_MAPILLARY.py --config 'nano' --weight '/kaggle/working/TwinLiteNetPlus/finetune/nano_weights.pth'\
                --data_root "/kaggle/input/datasets/kaggleprollc/mapillary-vistas-image-data-collection/Mapillary Vistas" \
                --num_workers 4 \
                --batch_size 16


Total network parameters: 33379
Driving Area Segment: mIOU(0.734)
Lane Line Segment: Acc(0.588) IOU(0.151)


## Small

### Valdiate with Base

In [9]:
!python val_MAPILLARY.py --config 'small' --weight '/kaggle/working/TwinLiteNetPlus/pretrained/small.pth' \
                --data_root "/kaggle/input/datasets/kaggleprollc/mapillary-vistas-image-data-collection/Mapillary Vistas" \
                --num_workers 4 \
                --batch_size 16


Total network parameters: 121552
Driving Area Segment: mIOU(0.697)
Lane Line Segment: Acc(0.533) IOU(0.055)


### Finetuning

In [10]:
# !python train_MAPILLARY.py \
#     --data_root "/kaggle/input/datasets/kaggleprollc/mapillary-vistas-image-data-collection/Mapillary Vistas" \
#     --hyp "./hyperparameters/twinlitev2_hyper.yaml" \
#     --savedir "./finetune_mapillary/small" \
#     --resume "/kaggle/working/TwinLiteNetPlus/pretrained/small.pth" \
#     --config "small" \
#     --max_epochs 40 \
#     --batch_size 32 \
#     --num_workers 4 \
#     --ema \
#     --verbose 

### Validate with Finetuned

In [11]:
!python val_MAPILLARY.py --config 'small' --weight '/kaggle/working/TwinLiteNetPlus/finetune/small_weights.pth' \
                --data_root "/kaggle/input/datasets/kaggleprollc/mapillary-vistas-image-data-collection/Mapillary Vistas" \
                --num_workers 4 \
                --batch_size 16


Total network parameters: 121552
Driving Area Segment: mIOU(0.747)
Lane Line Segment: Acc(0.600) IOU(0.172)


## Medium

### Validate with Base

In [12]:
!python val_MAPILLARY.py --config 'medium' --weight '/kaggle/working/TwinLiteNetPlus/pretrained/medium.pth' \
                --data_root "/kaggle/input/datasets/kaggleprollc/mapillary-vistas-image-data-collection/Mapillary Vistas" \
                --num_workers 4 \
                --batch_size 16


Total network parameters: 478876
Driving Area Segment: mIOU(0.703)
Lane Line Segment: Acc(0.536) IOU(0.059)


### Finetuning

In [13]:
# !python train_MAPILLARY.py \
#     --data_root "/kaggle/input/datasets/kaggleprollc/mapillary-vistas-image-data-collection/Mapillary Vistas" \
#     --hyp "./hyperparameters/twinlitev2_hyper.yaml" \
#     --savedir "./finetune_mapillary/medium" \
#     --resume "/kaggle/working/TwinLiteNetPlus/pretrained/medium.pth" \
#     --config "medium" \
#     --max_epochs 40 \
#     --batch_size 32 \
#     --num_workers 4 \
#     --ema \
#     --verbose

### Validate with Finetuned

In [14]:
!python val_MAPILLARY.py --config 'medium' --weight '/kaggle/working/TwinLiteNetPlus/finetune/medium_weights.pth' \
                --data_root "/kaggle/input/datasets/kaggleprollc/mapillary-vistas-image-data-collection/Mapillary Vistas" \
                --num_workers 4 \
                --batch_size 16


Total network parameters: 478876
Driving Area Segment: mIOU(0.763)
Lane Line Segment: Acc(0.619) IOU(0.200)


## Large

### Validate with Base

In [15]:
!python val_MAPILLARY.py --config 'large' --weight '/kaggle/working/TwinLiteNetPlus/pretrained/large.pth' \
                --data_root "/kaggle/input/datasets/kaggleprollc/mapillary-vistas-image-data-collection/Mapillary Vistas" \
                --num_workers 4 \
                --batch_size 16


Total network parameters: 1943911
Driving Area Segment: mIOU(0.704)
Lane Line Segment: Acc(0.542) IOU(0.065)


### Finetuning

In [16]:
# !python train_MAPILLARY.py \
#     --data_root "/kaggle/input/datasets/kaggleprollc/mapillary-vistas-image-data-collection/Mapillary Vistas" \
#     --hyp "./hyperparameters/twinlitev2_hyper.yaml" \
#     --savedir "./finetune_mapillary/large" \
#     --resume "/kaggle/working/TwinLiteNetPlus/pretrained/large.pth" \
#     --config "large" \
#     --max_epochs 40 \
#     --batch_size 32 \
#     --num_workers 4 \
#     --ema \
#     --verbose

### Validate with Fintune

In [17]:
!python val_MAPILLARY.py --config 'large' --weight '/kaggle/working/TwinLiteNetPlus/finetune/large_weights.pth' \
                --data_root "/kaggle/input/datasets/kaggleprollc/mapillary-vistas-image-data-collection/Mapillary Vistas" \
                --num_workers 4 \
                --batch_size 16


Total network parameters: 1943911
Driving Area Segment: mIOU(0.770)
Lane Line Segment: Acc(0.632) IOU(0.219)
